<h1>Chapter 7 - Retrieval</h1>
<i>Techniques to retrieve relevant information for your RAG system.</i>

<a href="https://learning.oreilly.com/library/view/rag-with-python/9798341600553/"><img src="https://img.shields.io/badge/O'Reilly-white.svg?logo=data:image/svg%2bxml;base64,PHN2ZyB3aWR0aD0iMzQiIGhlaWdodD0iMjciIHZpZXdCb3g9IjAgMCAzNCAyNyIgZmlsbD0ibm9uZSIgeG1sbnM9Imh0dHA6Ly93d3cudzMub3JnLzIwMDAvc3ZnIj4KPGNpcmNsZSBjeD0iMTMiIGN5PSIxNCIgcj0iMTEiIHN0cm9rZT0iI0Q0MDEwMSIgc3Ryb2tlLXdpZHRoPSI0Ii8+CjxjaXJjbGUgY3g9IjMwLjUiIGN5PSIzLjUiIHI9IjMuNSIgZmlsbD0iI0Q0MDEwMSIvPgo8L3N2Zz4K"></a>
<a href="https://github.com/polzerdo55862/RAG-with-Python-Cookbook"><img src="https://img.shields.io/badge/GitHub%20Repository-black?logo=github"></a>
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/polzerdo55862/RAG-with-Python-Cookbook/blob/main/ch07_retrieval/retrieval_techniques.ipynb)

---

This notebook is for Chapter 7 of the [RAG with Python Cookbook](https://learning.oreilly.com/library/view/rag-with-python/9798341600553/) book by [Dominik Polzer](https://www.linkedin.com/in/polzerdo/).

---

<a href="https://learning.oreilly.com/library/view/rag-with-python/9798341600553/">
  <img src="https://raw.githubusercontent.com/polzerdo55862/RAG-with-Python-Cookbook/main/rag_cookbook.png" width="350" />
</a>


## Prerequisits

If you are viewing this notebook on Google Colab (or any other cloud vendor), you need to uncomment and run the following codeblock to install the dependencies for this chapter.

In [5]:
!pip install psycopg2==2.9.10
!pip install requests==2.32.3
!pip install openai==1.82.1
!pip install python-dotenv==1.0.0


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


### Load sample files

This notebook uses sample Word and PDF files.

When running the notebook on Google Colab, uncomment the code below to download the `datasets` directory from the Github repo.

In [6]:
import sys

IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    !git clone --no-checkout https://github.com/polzerdo55862/RAG-with-Python-Cookbook.git
    %cd RAG-with-Python-Cookbook
    !git sparse-checkout init --cone
    !git sparse-checkout set datasets
    !git checkout
    !cp -r datasets /content/datasets
    print("\u2713 Datasets downloaded to /content/datasets")
else:
    print("\u26a0 Running locally. Using ../datasets/ directory")

⚠ Running locally. Using ../datasets/ directory


### Load secrets

If you run this code in Google Colab, save your OpenAI API key in the colab secrets and load it to the environmental variables.

In [7]:
import os
import sys
from dotenv import load_dotenv

# Check if running in Google Colab
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    try:
        from google.colab import userdata  # type: ignore

        os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
        os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")
        os.environ["GOOGLE_API_KEY"] = userdata.get("GOOGLE_API_KEY")
    except ModuleNotFoundError:
        pass
else:
    load_dotenv()

### 1 Optimizing Query Results using Metadata Filtering in PostgreSQL

In [8]:
import psycopg2
import json

POSTGRES_READY = False
conn = None
cur = None

try:
    conn = psycopg2.connect(
        dbname="rag_cookbook",
        user="rag_cookbook_user",
        password="rag_cookbook_user_pw",
        host="localhost",
        port="5432",
    )
    cur = conn.cursor()

    # Try enabling pgvector; if unavailable, keep notebook runnable without Docker.
    cur.execute("CREATE EXTENSION IF NOT EXISTS vector")

    cur.execute(
        """
        CREATE TABLE IF NOT EXISTS embeddings_table_with_metadata (
            id SERIAL PRIMARY KEY,
            text_chunk TEXT,
            embedding VECTOR(1536),
            metadata JSONB
        )
        """
    )
    conn.commit()
    POSTGRES_READY = True
    print("PostgreSQL with pgvector is ready.")
except Exception as exc:
    print(f"Skipping PostgreSQL vector setup: {exc}")
    if conn is not None:
        conn.rollback()

Skipping PostgreSQL vector setup: extension "vector" is not available
HINT:  The extension must first be installed on the system where PostgreSQL is running.



### 5.2 Enhancing Search Results by Extending the Original Query with Generated Pseudo-Documents

In [9]:
# Define text chunks and metadata
text_chunks = [
    {
        "text": "Roger Federer has won 20 Grand Slam titles in tennis.",
        "topic": "tennis",
    },
    {
        "text": "The FIFA World Cup is the most prestigious football tournament.",
        "topic": "football",
    },
    {
        "text": "Serena Williams is one of the greatest tennis players of all time.",
        "topic": "tennis",
    },
    {
        "text": "Lionel Messi has won multiple Ballon d'Or awards in football.",
        "topic": "football",
    },
]

if POSTGRES_READY and conn is not None and cur is not None:
    # Initialize OpenAI client
    client = OpenAI()

    # Insert text chunks with embeddings and metadata
    for chunk in text_chunks:
        response = client.embeddings.create(
            input=chunk["text"],
            model="text-embedding-3-small",
        )
        embedding = response.data[0].embedding

        metadata = {"topic": chunk["topic"]}
        cur.execute(
            "INSERT INTO embeddings_table_with_metadata (text_chunk, embedding, metadata) VALUES (%s, %s, %s)",
            (chunk["text"], embedding, json.dumps(metadata)),
        )

    conn.commit()
else:
    print("Skipping metadata insert because PostgreSQL/pgvector is not available.")

Skipping metadata insert because PostgreSQL/pgvector is not available.


In [10]:
hypothetical_documents if "hypothetical_documents" in globals() else []

[]

### 5.3 Improving Search Results with Multi-Query Retrieval

In [11]:
from openai import OpenAI
from pydantic import BaseModel
import os

client = OpenAI()

question = "What are the benefits of renewable energy?"

query_prompt = f"""You are an AI language model assistant. Your task is
to create three alternative versions of the provided user query to
enhance the retrieval of relevant documents from a vector database.
By offering diverse variations of the query, your goal is to help
mitigate the limitations of distance-based similarity search. Provide
these alternative queries, each on a new line.

Original query: {question}
"""

# send the query prompt to OpenAI
class QueryVariations(BaseModel):
    queries: list[str]

completion = client.beta.chat.completions.parse(
    model="gpt-5-mini",
    messages=[
        {
            "role": "user",
            "content": query_prompt,
        },
    ],
    response_format=QueryVariations,
)

queries = completion.choices[0].message.parsed.queries
queries

['What are the environmental, economic, and social advantages of using renewable energy sources?',
 'How does switching to renewable energy (solar, wind, hydro) benefit public health, the economy, and national energy security?',
 'List the key benefits of renewable energy, including cost savings, emissions reduction, job creation, and improved grid resilience.']

### 5.4 Addressing Complex Requests by Designing a Query Routing System

In [12]:
# Query the table with metadata filtering
query = "Who is the best player?"
topic_filter = "football"

if POSTGRES_READY and conn is not None and cur is not None:
    response = client.embeddings.create(
        input=query,
        model="text-embedding-3-small",
    )
    query_embedding = response.data[0].embedding

    cur.execute(
        """
        SELECT text_chunk, 1 - (embedding <=> %s::vector) AS similarity
        FROM embeddings_table_with_metadata
        WHERE metadata->>'topic' = %s
        ORDER BY similarity DESC
        LIMIT 5
        """,
        (query_embedding, topic_filter),
    )
    results = cur.fetchall()
else:
    results = []
    print("Skipping metadata filtering query because PostgreSQL/pgvector is not available.")

results

Skipping metadata filtering query because PostgreSQL/pgvector is not available.


[]

### 5.5 Increasing Search Efficiency by Designing an Auto-Merging Retriever (aka Parent Document Retriever)

In [13]:
from pydantic import BaseModel
from openai import OpenAI

user_query = "What is the revenue of Company X in 2024?"

client = OpenAI()

class HypotheticalDocuments(BaseModel):
    documents: list[str]

prompt = f"""
You are an AI assistant. Based on the user query below, generate
three hypothetical text chunks that contain relevant information to
answer the query.
"""

completion = client.beta.chat.completions.parse(
    messages=[
        {"role": "system", "content": prompt},
        {"role": "user", "content": user_query},
    ],
    model="gpt-5-mini",
    response_format=HypotheticalDocuments,
)

hypothetical_documents = completion.choices[0].message.parsed.documents
hypothetical_documents

['Company X Press Release — February 14, 2025: "Company X Reports Full-Year 2024 Results" — For the fiscal year ended December 31, 2024, Company X reported consolidated revenue of $1.25 billion (USD), a 6% increase compared with $1.18 billion for the prior year. The release states the figure is on a GAAP basis and includes revenues from all continuing operations.',
 'Excerpt from Company X Form 10‑K (filed March 1, 2025) — Item 6. Selected Financial Data: "Total revenues, net — Year ended December 31, 2024: $1,250,000,000. The Company’s consolidated financial statements present all amounts in U.S. dollars. The 2024 revenue figure reflects continued operations and excludes amounts from the divested Widget division sold in Q3 2023, per Management’s discussion and analysis."',
 'Equity Research Note — Capital Markets, March 3, 2025: "Company X reported FY2024 revenue of $1.25 billion (USD). Segment breakdown provided by management: Product Solutions revenue $800 million; Services & Suppor

### 5.6 Increasing Search Results by Designing a Sentence Window Retriever

In [14]:
if POSTGRES_READY and conn is not None and cur is not None:
    cur.execute(
        """
        CREATE TABLE sentence_window_retriever_text_chunks (
            chunk_id SERIAL PRIMARY KEY,
            chunk TEXT,
            chunk_embedding VECTOR(1536)
        )
        """
    )
    conn.commit()
else:
    print("Skipping sentence-window table creation because PostgreSQL/pgvector is not available.")

Skipping sentence-window table creation because PostgreSQL/pgvector is not available.


### 5.7 Improving Search Accuracy with Reranking Methods

In [15]:
from pydantic import BaseModel
from openai import OpenAI

client = OpenAI()

user_queries = [
    {
        "query": "Who is the all-time top scorer in the FIFA World Cup?",
        "selected_data_source": None,
    },
    {
        "query": "What are the four Grand Slam tennis tournaments?",
        "selected_data_source": None,
    },
    {
        "query": "Did Manchester United win their last game?",
        "selected_data_source": None,
    },
]

prompt = f"""
You are an expert at routing a user question to the appropriate
data source. Given a user question choose which of the data sources
in list_of_data_sources is the best to answer the question.
"""

from typing import Literal
from pydantic import Field

class RouterDecision(BaseModel):
    data_source: Literal[
        "general_football_knowledge",
        "general_tennis_knowledge",
        "latest_football_results_sql",
    ] = Field(
        ...,
        description="The best data source to answer the question."
    )

for user_query in user_queries:
    completion = client.beta.chat.completions.parse(
        messages=[
            {"role": "system", "content": prompt},
            {"role": "user", "content": user_query["query"]},
        ],
        model="gpt-5-mini",
        response_format=RouterDecision,
    )
    user_query["selected_data_source"] = completion.choices[
        0
    ].message.parsed.data_source

completion

ParsedChatCompletion[RouterDecision](id='chatcmpl-DHAWKCHMy4wlFcbJTmwqyua5U5Fd8', choices=[ParsedChoice[RouterDecision](finish_reason='stop', index=0, logprobs=None, message=ParsedChatCompletionMessage[RouterDecision](content='{"data_source":"latest_football_results_sql"}', refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=None, parsed=RouterDecision(data_source='latest_football_results_sql')))], created=1772984388, model='gpt-5-mini-2025-08-07', object='chat.completion', service_tier='default', system_fingerprint=None, usage=CompletionUsage(completion_tokens=87, prompt_tokens=126, total_tokens=213, completion_tokens_details=CompletionTokensDetails(accepted_prediction_tokens=0, audio_tokens=0, reasoning_tokens=64, rejected_prediction_tokens=0), prompt_tokens_details=PromptTokensDetails(audio_tokens=0, cached_tokens=0)))

### 5.8 Decomposing Complex Queries into Multiple Sub-Queries

In [16]:
from pydantic import BaseModel 
from typing import Optional 
from openai import OpenAI

class Question(BaseModel):
    question: str
    answer: Optional[str] = None

class Questions(BaseModel): 
    questions: list[Question]

splitter_prompt = """
You are a helpful assistant for a RAG chatbot.

Your job is to break down complex questions into simpler ones that 
are easy to answer. When the answers to these simpler questions are 
combined, they should fully answer the original question. If the 
question is already simple, leave it as it is. Handle one question 
at a time.

Example:
    1. Query: Did Microsoft or Google make more money last year?

Decomposed Questions:
    1. How much profit did Microsoft make last year?
    2. How much profit did Google make last year? 
"""

client = OpenAI()

query = (
    "What are the benefits of renewable energy compared to "
    "fossil fuels?"
)

completion = client.beta.chat.completions.parse(
    model="gpt-5-mini",
        messages=[
        {"role": "system", "content": splitter_prompt}, 
        {"role": "user", "content": query},
    ], 
    response_format=Questions,
)

decomposed_questions = completion.choices[0].message.parsed.questions
decomposed_questions


[Question(question='How do greenhouse gas emissions from renewable energy compare with fossil fuels over their lifecycles?', answer=None),
 Question(question='How do renewable energy sources affect air and water pollution relative to fossil fuels?', answer=None),
 Question(question='What are the public health benefits of using renewables instead of fossil fuels?', answer=None),
 Question(question='How do the costs of renewable energy (capital costs, operating costs, levelized cost of energy) compare to fossil fuels now and in projected future trends?', answer=None),
 Question(question='How does deployment of renewable energy affect jobs, local economic development, and industry growth compared with fossil fuel industries?', answer=None),
 Question(question='What energy security and geopolitical benefits do renewables offer compared with dependence on fossil fuel imports and supply chains?', answer=None),
 Question(question='How do renewables impact grid resilience and reliability, incl